# Generalization to Unseen Airfoil Geometries

How well does the model predict lift, drag, and pitching moment for an unseen airfoil? The geometry benchmarks keep all operating conditions for equivalent shapes in the same split.

## Setup and Configuration

In [1]:
# Set TensorFlow logging and numerical options
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

In [2]:
# Load the libraries and configure the experiment
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from keras import Input, Sequential, backend, utils
from keras.callbacks import ReduceLROnPlateau
from keras.layers import Dense
from keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "notebooks").is_dir() and (p / "results").is_dir()
)
sys.path.insert(0, str(ROOT / "notebooks"))
experiment_results = __import__("experiment_results")
geometry_splits = __import__("geometry_splits")

ARCHITECTURE = (512, 256, 128)
N_RUNS = 20
EPOCHS = 50
TARGETS = ["CL", "CD", "Cm"]

In [3]:
# Format coefficient tables consistently
def style_table(table):
    return table.style.set_table_styles(
        [
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "td", "props": [("text-align", "left")]},
        ]
    ).format(precision=6)


def show_table(table):
    display(style_table(table))


# Group coefficient scores under each metric
def show_metrics(results, index, include_validation=False):
    columns = {"Test rows": ("", "Test rows")}
    if include_validation:
        columns["Mean normalized validation MSE"] = ("Validation", "Normalized MSE")
    for metric, label in (("rmse", "RMSE"), ("r2", "R²")):
        for target in TARGETS:
            coefficient = f"<i>C<sub>{target[1:]}</sub></i>"
            columns[f"{target}_{metric}"] = (label, coefficient)

    table = pd.DataFrame(results).set_index(index)[list(columns)]
    table.columns = pd.MultiIndex.from_tuples(columns.values())
    show_table(table)

## Match Cases and Split by Geometry

In [4]:
# Load matching cases and assign a group to each geometry
frames = {}
matched = {}
for family in geometry_splits.PARAMETERS:
    frames[family] = {}
    for n in geometry_splits.RESOLUTIONS:
        name = f"{family}_{n:02}"
        frames[family][n] = pd.read_csv(
            experiment_results.dataset_path(ROOT, name)
        ).assign(dataset=name)
    matched[family] = geometry_splits.match_cases(frames[family], family)

pooled_manifest = experiment_results.dataset_manifest(ROOT, ["NACA4D_10", "NACA5D_10"])
groups = geometry_splits.geometry_groups(matched)
assignment = {
    family: geometry_splits.group_split(labels) for family, labels in groups.items()
}
pooled_groups = pd.Series([label for labels in groups.values() for label in labels])
pooled_assignment = geometry_splits.group_split(pooled_groups)
split_rows = []
for family, labels, splits in [
    ("4-digit", groups["NACA4D"], assignment["NACA4D"]),
    ("5-digit", groups["NACA5D"], assignment["NACA5D"]),
    ("Pooled (10 points)", pooled_groups, pooled_assignment),
]:
    for label, keys in splits.items():
        split_rows.append(
            (family, label.title(), pd.Series(labels).isin(keys).sum(), len(keys))
        )
split_summary = pd.DataFrame(
    split_rows, columns=["Family", "Split", "Rows", "Geometry groups"]
).set_index(["Family", "Split"])
show_table(split_summary)

## Training and Evaluation

In [5]:
# Configure training and score each coefficient
def make_reduce_lr():
    return ReduceLROnPlateau(monitor="val_loss", factor=0.9, patience=5, min_lr=0.00001)


def coefficient_metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted, multioutput="raw_values")
    r2 = r2_score(actual, predicted, multioutput="raw_values")
    return {
        **{f"{target}_mse": mse[j] for j, target in enumerate(TARGETS)},
        **{f"{target}_rmse": np.sqrt(mse[j]) for j, target in enumerate(TARGETS)},
        **{f"{target}_r2": r2[j] for j, target in enumerate(TARGETS)},
    }


# Train and evaluate the fixed architecture on one split
def evaluate(name, resolution, splits, datasets, protocol):
    columns = geometry_splits.geometry_columns(resolution) + geometry_splits.CONDITIONS
    train, validation, test = (
        splits[label] for label in ("train", "validation", "test")
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[columns])
    X_val = scaler.transform(validation[columns])
    X_test = scaler.transform(test[columns])
    y_train, y_val, y_test = (frame[TARGETS] for frame in (train, validation, test))
    history, validation_runs, test_runs = [], [], []
    for i in range(N_RUNS):
        print(f"Run {i + 1}/{N_RUNS} for {name}", flush=True)
        backend.clear_session()
        utils.set_random_seed(42 + i)
        model = Sequential(
            [Input(shape=(len(columns),))]
            + [Dense(size, activation="relu") for size in ARCHITECTURE]
            + [Dense(len(TARGETS), activation="linear")]
        )
        model.compile(
            optimizer=Adam(learning_rate=5e-4, beta_1=0.9, beta_2=0.999),
            loss="mse",
        )
        run = model.fit(
            X_train,
            y_train,
            epochs=EPOCHS,
            batch_size=128,
            validation_data=(X_val, y_val),
            callbacks=[make_reduce_lr()],
            verbose=0,
        )
        history.append(run.history)
        validation_runs.append(
            coefficient_metrics(y_val, model.predict(X_val, verbose=0))
        )
        predictions = model.predict(X_test, verbose=0)
        test_runs.append(coefficient_metrics(y_test, predictions))
        if "exclusive" in splits:
            mask = test.index.isin(splits["exclusive"].index)
            scores = coefficient_metrics(y_test.loc[mask], predictions[mask])
            test_runs[-1].update(
                {f"exclusive_{key}": value for key, value in scores.items()}
            )

    metrics = pd.DataFrame(test_runs)
    validation_metrics = pd.DataFrame(validation_runs)
    settings = experiment_results.training_settings(N_RUNS, EPOCHS, y_train, protocol)
    settings["row_identifier"] = "[dataset, zero-based source CSV row]"
    row_ids = {
        label: list(frame[["dataset", "csv_row"]].itertuples(index=False, name=None))
        for label, frame in splits.items()
    }
    architecture = "M_" + "_".join(map(str, ARCHITECTURE))
    experiment_results.export_results(
        ROOT,
        name,
        datasets,
        row_ids,
        {architecture: metrics},
        {architecture: validation_metrics},
        {architecture: history},
        settings,
    )
    normalized_mse = (
        pd.DataFrame(
            {
                target: validation_metrics[f"{target}_mse"]
                / settings["training_target_variance"][target]
                for target in TARGETS
            }
        )
        .to_numpy()
        .mean()
    )
    means = metrics.mean().to_dict()
    return {
        "Experiment": name,
        "Test rows": len(test),
        "Mean normalized validation MSE": normalized_mse,
        **means,
    }

## Coordinate Resolution Comparison

Each family uses the same cases and split assignments at all three resolutions.

In [6]:
# Reuse each family's split assignments across all resolutions
results = []
for family, datasets in matched.items():
    family_label = "4-digit" if family == "NACA4D" else "5-digit"
    for n, frame in datasets.items():
        name = f"{family}_{n:02}"
        labels = pd.Series(groups[family], index=frame.index)
        splits = {
            label: frame.loc[labels.isin(keys)]
            for label, keys in assignment[family].items()
        }
        manifest = experiment_results.dataset_manifest(ROOT, [name])
        result = evaluate(
            f"geometry-{name}", n, splits, manifest, "geometry-grouped-matched"
        )
        result["Family"] = family_label
        result["Coordinate points"] = n
        results.append(result)

show_metrics(results, ["Family", "Coordinate points"], include_validation=True)

Run 1/20 for geometry-NACA4D_05
Run 2/20 for geometry-NACA4D_05
Run 3/20 for geometry-NACA4D_05
Run 4/20 for geometry-NACA4D_05
Run 5/20 for geometry-NACA4D_05
Run 6/20 for geometry-NACA4D_05
Run 7/20 for geometry-NACA4D_05
Run 8/20 for geometry-NACA4D_05
Run 9/20 for geometry-NACA4D_05
Run 10/20 for geometry-NACA4D_05
Run 11/20 for geometry-NACA4D_05
Run 12/20 for geometry-NACA4D_05
Run 13/20 for geometry-NACA4D_05
Run 14/20 for geometry-NACA4D_05
Run 15/20 for geometry-NACA4D_05
Run 16/20 for geometry-NACA4D_05
Run 17/20 for geometry-NACA4D_05
Run 18/20 for geometry-NACA4D_05
Run 19/20 for geometry-NACA4D_05
Run 20/20 for geometry-NACA4D_05
Run 1/20 for geometry-NACA4D_10
Run 2/20 for geometry-NACA4D_10
Run 3/20 for geometry-NACA4D_10
Run 4/20 for geometry-NACA4D_10
Run 5/20 for geometry-NACA4D_10
Run 6/20 for geometry-NACA4D_10
Run 7/20 for geometry-NACA4D_10
Run 8/20 for geometry-NACA4D_10
Run 9/20 for geometry-NACA4D_10
Run 10/20 for geometry-NACA4D_10
Run 11/20 for geometry-NACA4

## Pooled Geometry Benchmark

The 10-point model is tested on geometries held out across both families.

In [7]:
# Test the pooled model on geometries excluded from training in both families
pooled = pd.concat([datasets[10] for datasets in matched.values()], ignore_index=True)
splits = {
    label: pooled.loc[pooled_groups.isin(keys)]
    for label, keys in pooled_assignment.items()
}
result = evaluate(
    "geometry-pooled-10", 10, splits, pooled_manifest, "geometry-grouped-pooled"
)
result["Family"] = "Pooled"
result["Coordinate points"] = 10
show_metrics([result], ["Family", "Coordinate points"], include_validation=True)

Run 1/20 for geometry-pooled-10
Run 2/20 for geometry-pooled-10
Run 3/20 for geometry-pooled-10
Run 4/20 for geometry-pooled-10
Run 5/20 for geometry-pooled-10
Run 6/20 for geometry-pooled-10
Run 7/20 for geometry-pooled-10
Run 8/20 for geometry-pooled-10
Run 9/20 for geometry-pooled-10
Run 10/20 for geometry-pooled-10
Run 11/20 for geometry-pooled-10
Run 12/20 for geometry-pooled-10
Run 13/20 for geometry-pooled-10
Run 14/20 for geometry-pooled-10
Run 15/20 for geometry-pooled-10
Run 16/20 for geometry-pooled-10
Run 17/20 for geometry-pooled-10
Run 18/20 for geometry-pooled-10
Run 19/20 for geometry-pooled-10
Run 20/20 for geometry-pooled-10


## Cross-Family Transfer

Each direction scores the full target family and then the subset with shared profiles excluded.

In [8]:
# Repeat each original transfer split and score both target populations
transfer_results = []
families = list(geometry_splits.PARAMETERS)
for source_family, target_family in (families, families[::-1]):
    source = frames[source_family][10].assign(csv_row=lambda frame: frame.index)
    target = frames[target_family][10].assign(csv_row=lambda frame: frame.index)
    source = source.sample(frac=1, random_state=42)
    target = target.sample(frac=1, random_state=42)
    train, validation = train_test_split(source, train_size=0.8, random_state=42)
    exclusive = geometry_splits.exclusive_target(
        source, target, geometry_splits.geometry_columns(10)
    )
    splits = {
        "train": train,
        "validation": validation,
        "test": target,
        "exclusive": target.loc[exclusive],
    }
    name = f"geometry-{source_family}_10-to-{target_family}_10"
    result = evaluate(
        name, 10, splits, pooled_manifest, "cross-family-shared-profile-check"
    )
    direction = (
        "4-digit → 5-digit" if source_family == "NACA4D" else "5-digit → 4-digit"
    )
    transfer_results.append(
        {"Direction": direction, "Test set": "Full family", **result}
    )
    transfer_results.append(
        {
            "Direction": direction,
            "Test set": "Shared profiles excluded",
            "Test rows": int(exclusive.sum()),
            **{
                f"{target}_{metric}": result[f"exclusive_{target}_{metric}"]
                for target in TARGETS
                for metric in ("rmse", "r2")
            },
        }
    )

show_metrics(transfer_results, ["Direction", "Test set"])

Run 1/20 for geometry-NACA4D_10-to-NACA5D_10
Run 2/20 for geometry-NACA4D_10-to-NACA5D_10
Run 3/20 for geometry-NACA4D_10-to-NACA5D_10
Run 4/20 for geometry-NACA4D_10-to-NACA5D_10
Run 5/20 for geometry-NACA4D_10-to-NACA5D_10
Run 6/20 for geometry-NACA4D_10-to-NACA5D_10
Run 7/20 for geometry-NACA4D_10-to-NACA5D_10
Run 8/20 for geometry-NACA4D_10-to-NACA5D_10
Run 9/20 for geometry-NACA4D_10-to-NACA5D_10
Run 10/20 for geometry-NACA4D_10-to-NACA5D_10
Run 11/20 for geometry-NACA4D_10-to-NACA5D_10
Run 12/20 for geometry-NACA4D_10-to-NACA5D_10
Run 13/20 for geometry-NACA4D_10-to-NACA5D_10
Run 14/20 for geometry-NACA4D_10-to-NACA5D_10
Run 15/20 for geometry-NACA4D_10-to-NACA5D_10
Run 16/20 for geometry-NACA4D_10-to-NACA5D_10
Run 17/20 for geometry-NACA4D_10-to-NACA5D_10
Run 18/20 for geometry-NACA4D_10-to-NACA5D_10
Run 19/20 for geometry-NACA4D_10-to-NACA5D_10
Run 20/20 for geometry-NACA4D_10-to-NACA5D_10
Run 1/20 for geometry-NACA5D_10-to-NACA4D_10
Run 2/20 for geometry-NACA5D_10-to-NACA4D_10